# Tratativas

### Imports:

In [31]:
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
import pandas as pd

### Criação do banco de dados .csv

In [33]:
nome_arquivo_excel = '../dados/base_quedas_AACD_anonimizada.xlsx'
nome_arquivo_csv = '../dados/base_quedas_AACD_anonimizada.csv'

try:
    # Carregar as duas abas do Excel
    print("Lendo as abas do Excel...")
    df_60_mais = pd.read_excel(nome_arquivo_excel, sheet_name='60+')
    df_60_menos = pd.read_excel(nome_arquivo_excel, sheet_name='60-')

    # Função para juntar as linhas de 2 em 2
    def consertar_linhas_duplas(df):
        # Agrupa de 2 em 2 linhas e pega o primeiro valor não nulo
        df_agrupado = df.groupby(df.index // 2).first()
        return df_agrupado

    # Aplicar a correção nas duas tabelas
    print("Corrigindo o cadastro feito em duas linhas...")
    df_60_mais_corrigido = consertar_linhas_duplas(df_60_mais)
    df_60_menos_corrigido = consertar_linhas_duplas(df_60_menos)

    # Juntar as duas tabelas em uma só
    print("Unindo as duas tabelas...")
    df_final = pd.concat([df_60_mais_corrigido, df_60_menos_corrigido], ignore_index=True)

    # ---------------------------------------------------------
    print("Corrigindo a numeração dos IDs...")
    df_final['ID'] = range(1, len(df_final) + 1)
    # ---------------------------------------------------------

    # Salvar o resultado no arquivo .csv
    df_final.to_csv(nome_arquivo_csv, index=False, encoding='utf-8-sig')
    
    print(f"Sucesso! Arquivo '{nome_arquivo_csv}' criado com IDs sequenciais.")

except FileNotFoundError:
    print(f"Erro: O arquivo '{nome_arquivo_excel}' não foi encontrado na pasta.")
except Exception as e:
    print(f"Ocorreu um erro: {e}")

Lendo as abas do Excel...
Corrigindo o cadastro feito em duas linhas...
Unindo as duas tabelas...
Corrigindo a numeração dos IDs...
Sucesso! Arquivo '../dados/base_quedas_AACD_anonimizada.csv' criado com IDs sequenciais.


### Correções do nome das colunas:

In [34]:
df = pd.read_csv('../dados/base_quedas_AACD_anonimizada.csv')

colunas_com_interrogacao = [
    'Clínica/Dx',
    'Aditamento 10m',
    'Status funcional',
    'Diagnóstico Funcional',
]
for coluna in colunas_com_interrogacao:
    df[coluna] = df[coluna].astype('string').str.replace('?', '', regex=False)

colunas = df.columns.tolist()
print(colunas)

['ID', 'Sexo', 'Idade ', 'Clínica/Dx', 'Diagnóstico Funcional', 'Polifarmácia?', 'Suporte social (terapêutico)', 'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)', 'Cognição         (10-CS)', 'Nº quedas ao chão', 'Queda último mês', 'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D', 'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m', 'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)', 'Reação (blazepod) DT', 'Custo dupla tarefa']


In [35]:
df.rename(columns={'Polifarmácia?': 'Polifarmácia'}, inplace=True)
print(df.columns)

Index(['ID', 'Sexo', 'Idade ', 'Clínica/Dx', 'Diagnóstico Funcional',
       'Polifarmácia', 'Suporte social (terapêutico)',
       'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)',
       'Cognição         (10-CS)', 'Nº quedas ao chão', 'Queda último mês',
       'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D',
       'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m',
       'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)',
       'Reação (blazepod) DT', 'Custo dupla tarefa'],
      dtype='str')


In [36]:
df.rename(columns={'Idade ': 'Idade'}, inplace=True)
print(df.columns)

Index(['ID', 'Sexo', 'Idade', 'Clínica/Dx', 'Diagnóstico Funcional',
       'Polifarmácia', 'Suporte social (terapêutico)',
       'Suporte social (domiciliar)', 'Status funcional', 'Cognição (MOCA)',
       'Cognição         (10-CS)', 'Nº quedas ao chão', 'Queda último mês',
       'Sentar/levantar 30s', 'Ponte', 'Dinamômetro', 'Panturrilha D',
       'Panturrilha E', 'Braço D', 'Braço E', '10 metros', 'Aditamento 10m',
       'CIF equilíbrio', 'Posição equilíbrio', 'Reação (blazepod)',
       'Reação (blazepod) DT', 'Custo dupla tarefa'],
      dtype='str')


### Correções nas colunas

- Apaga qualquer espaço em branco antes ou depois da palavra, transforma 'sim ' em 'sim'.

In [37]:
df['Polifarmácia'] = (
    df['Polifarmácia']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA})
    .fillna('não')
    .astype('category')
)

print(df['Polifarmácia'].dtype)
print(df['Polifarmácia'].value_counts())

category
Polifarmácia
não    143
sim    139
Name: count, dtype: int64


- Remove a marca "?" antes da conversão numérica. Idades inválidas permanecem ausentes, sem imputação.

In [38]:
df['Idade'] = df['Idade'].astype('string').str.replace('?', '', regex=False).str.strip()

In [39]:
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
# Calcula a média das idades (desconsiderando valores nulos)
df['Idade'] = pd.to_numeric(df['Idade'], errors='coerce').astype('Int64')
media_idade = df['Idade'].mean()
media_idade = int(round(media_idade))

df.loc[df['ID'] == 275, 'Idade'] = media_idade

#### Correções de tipagem

In [40]:
df.tail(1)

,ID,Sexo,Idade,Clínica/Dx,Diagnóstico Funcional,Polifarmácia,Suporte social (terapêutico),Suporte social (domiciliar),Status funcional,Cognição (MOCA),...,Panturrilha E,Braço D,Braço E,10 metros,Aditamento 10m,CIF equilíbrio,Posição equilíbrio,Reação (blazepod),Reação (blazepod) DT,Custo dupla tarefa
281,282,M,27,AMP,<NA>,não,namorada,NaN,marcha comunitária,NaN,...,NaN,NaN,NaN,7.0,Muletas canadenses,0,Ortostatismo,51.0,29.0,NaN


In [63]:
df.dtypes

ID                                 int64
Sexo                            category
Idade                              Int64
Clínica/Dx                        string
Diagnóstico Funcional             string
Polifarmácia                    category
Suporte social (terapêutico)      string
Suporte social (domiciliar)       string
Status funcional                  string
Cognição (MOCA)                    Int64
Cogniçao (10-CS)                   Int64
Nº quedas ao chão                  Int64
Queda último mês                category
Sentar/levantar 30s                Int64
Ponte                              Int64
Dinamômetro                        Int64
Panturrilha D                    float64
Panturrilha E                    float64
Braço D                          float64
Braço E                          float64
10 metros                        float64
Aditamento 10m                    string
CIF equilíbrio                     Int64
Posição equilíbrio              category
Reação (blazepod

In [42]:
df = df.rename(columns={'Cognição         (10-CS)': 'Cogniçao (10-CS)'})

In [43]:
df['Reação (blazepod)'] = pd.to_numeric(df['Reação (blazepod)'], errors='coerce').astype('Int64')
df['Reação (blazepod) DT'] = pd.to_numeric(df['Reação (blazepod) DT'], errors='coerce').astype('Int64')
df['CIF equilíbrio'] = pd.to_numeric(df['CIF equilíbrio'], errors='coerce').astype('Int64')
df['Ponte'] = pd.to_numeric(df['Ponte'], errors='coerce').astype('Int64')
df['Sentar/levantar 30s'] = pd.to_numeric(df['Sentar/levantar 30s'], errors='coerce').astype('Int64')
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')
df['Cogniçao (10-CS)'] = pd.to_numeric(df['Cogniçao (10-CS)'], errors='coerce').astype('Int64')
df['Cognição (MOCA)'] = pd.to_numeric(df['Cognição (MOCA)'], errors='coerce').astype('Int64')
df['Dinamômetro'] = pd.to_numeric(df['Dinamômetro'], errors='coerce').round().astype('Int64')

### Cálculo da velocidade no teste de 10 metros

A coluna `10 metros` registra o tempo em segundos. A velocidade é calculada em metros por segundo (m/s) pela distância de 10 metros dividida pelo tempo. Tempos ausentes ou não positivos geram velocidade ausente.

In [44]:
df['10 metros'] = pd.to_numeric(df['10 metros'], errors='coerce')
df['Velocidade (m/s)'] = 10 / df['10 metros'].where(df['10 metros'] > 0)

In [45]:
df['Sexo'] = df['Sexo'].astype('category')
df['Posição equilíbrio'] = df['Posição equilíbrio'].astype('category')
df['Panturrilha D'] = pd.to_numeric(df['Panturrilha D'], errors='coerce').astype('float')

In [46]:
df.isnull().sum()

ID                                0
Sexo                              0
Idade                             0
Clínica/Dx                        0
Diagnóstico Funcional           213
Polifarmácia                      0
Suporte social (terapêutico)      5
Suporte social (domiciliar)      18
Status funcional                 18
Cognição (MOCA)                 206
Cogniçao (10-CS)                216
Nº quedas ao chão                 8
Queda último mês                  7
Sentar/levantar 30s             170
Ponte                           126
Dinamômetro                       5
Panturrilha D                   109
Panturrilha E                   120
Braço D                         249
Braço E                         257
10 metros                        46
Aditamento 10m                   52
CIF equilíbrio                    7
Posição equilíbrio                6
Reação (blazepod)                 7
Reação (blazepod) DT             15
Custo dupla tarefa              282
Velocidade (m/s)            

### Tratamento conservador das colunas de suporte e status

Padroniza caixa e espaços, converte células vazias e o valor "." em ausentes, sem agrupar descrições diferentes.

In [47]:
colunas_texto_livre = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
    'Status funcional',
]

for coluna in colunas_texto_livre:
    df[coluna] = (
        df[coluna]
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
        .replace(['', '.'], pd.NA)
    )

colunas_suporte = [
    'Suporte social (terapêutico)',
    'Suporte social (domiciliar)',
]
df[colunas_suporte] = df[colunas_suporte].fillna('não informado')

In [48]:
pd.DataFrame({
    'valores_distintos': df[colunas_texto_livre].nunique(dropna=True),
    'ausentes': df[colunas_texto_livre].isna().sum(),
})

,valores_distintos,ausentes
Suporte social (terapêutico),49,0
Suporte social (domiciliar),62,0
Status funcional,122,18


### Salva as alterações diretamente no arquivo .csv

Facilita a visualização diretamente no arquivo

In [64]:
df['Polifarmácia'] = (
    df['Polifarmácia']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA, 'nan': pd.NA, 'NaN': pd.NA, 'None': pd.NA, 'null': pd.NA, '<NA>': pd.NA})
    .fillna('não')
    .astype('category')
)

df.to_csv('../dados/base_quedas_AACD_anonimizada.csv', index=False, encoding='utf-8-sig')
print("Arquivo salvo com sucesso!")

Arquivo salvo com sucesso!


##### Tratamento de nulos

In [50]:
df['Queda último mês'].isnull().sum()

np.int64(7)

In [51]:
df['Queda último mês'] = df['Queda último mês'].astype(str).str.strip()

df['Queda último mês'] = df['Queda último mês'].replace({
    '0': 'não',
    '2': 'sim',
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Queda último mês'] = df['Queda último mês'].fillna('não informado')
df['Queda último mês'] = df['Queda último mês'].astype('category')

In [52]:
print(df['Queda último mês'].isna().sum())

print(df['Queda último mês'].value_counts(dropna=False))

0
Queda último mês
não              226
sim               49
não informado      7
Name: count, dtype: int64


In [53]:
df['Nº quedas ao chão'] = df['Nº quedas ao chão'].astype(str).str.strip()

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].replace({
    '': 'não informado',
    'nan': 'não informado',
    'NaN': 'não informado',
    'null': 'não informado',
    'None': 'não informado'
})

df['Nº quedas ao chão'] = df['Nº quedas ao chão'].fillna('não informado')

##### Padronizar os registros

In [54]:
df['Clínica/Dx'] = df['Clínica/Dx'].astype('string').str.strip().str.upper()

# Dicionário de substituição dos diagnósticos
diagnosticos = {
    'AMP': 'AMP - Amputação',
    'AMP BILATERAL': 'AMP Bilateral - Amputação bilateral',
    'AMP LEA': 'AMP LEA - Amputação e Lesão Encefálica Adquirida',
    'AVC': 'AVC - Acidente Vascular Cerebral',
    'DNM': 'DNM - Doença Neuromuscular',
    'ELA': 'ELA - Esclerose Lateral Amiotrófica',
    'LEA': 'LEA - Lesão Encefálica Adquirida',
    'LM': 'LM - Lesão Medular',
    'PARKISON': 'Parkinson',
    'PARKINSON': 'Parkinson',
    'PC': 'PC - Paralisia Cerebral',
    'POLIO': 'Poliomielite',
    'POLIO?': 'Poliomielite',
    'MFC': 'MFC - Malformações Congênitas',
    'Mielo': 'Mielomeningocele',
    'EM': 'EM - Esclerose Múltipla',
    'LEA/LM': 'LEA/LM - Lesão Encefálica Adquirida e Lesão Medular'
}
df['Clínica/Dx'] = df['Clínica/Dx'].replace(diagnosticos)

### Tratativa da coluna status funcional

In [55]:
# vendo quantidade de valores nulos na coluna 'Status funcional'
df['Status funcional'].isnull().sum()

# Substitui os vazios (e o 'nan' que virou texto) por "não informado"
df['Status funcional'] = df['Status funcional'].fillna('não informado')

# substitutindo o texto cadeira de rodas por cadeirante
df['Status funcional'] = df['Status funcional'].replace('cadeira de rodas', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('deslocamento por cr', 'cadeirante')
df['Status funcional'] = df['Status funcional'].replace('marca comunitária', 'marcha comunitária')

#padronizando os textos para minúsculo
df['Status funcional'] = df['Status funcional'].str.lower()

#corrigindo os textos com erros de digitação
df['Status funcional'] = df['Status funcional'].replace({
    'marcha comunitária i': 'marcha comunitária independente',
    'cadeira de rodas ': 'cadeirante',
    'marhca comunitária com andador': 'marcha comunitária com andador',
    'marhca comunitária com bastão e supervisão': 'marcha comunitária com bastão e supervisão',
    'marcha comunitária i': 'marcha comunitária independente',
    
})

# removendo os '?' que ficaram no meio do texto
df['Status funcional'] = df['Status funcional'].str.replace('?', '', regex=False)

print(df['Status funcional'].value_counts())



Status funcional
cadeirante                                    59
não informado                                 18
marcha domiciliar com andador                 18
marcha comunitária com muletas canadenses     10
marcha comunitária                             6
                                              ..
marcha terapêutica com andador hospitalar      1
marcha domiciliar com apoio em móveis          1
marcha com apoio em terceiros                  1
marcha com andador reciprocado                 1
marcha domiciliar com andador de antebraço     1
Name: count, Length: 118, dtype: Int64


In [56]:
#visualizando os a colunas 'status funcional'
df[['Status funcional']]


,Status funcional
0,marcha domiciliar com supervisão
1,não informado
2,não informado
3,não informado
4,deambuladora comunitária independente
...,...
277,cadeirante
278,marcha domiciliar com andador de antebraço
279,marcha domiciliar
280,não informado


In [57]:
# substituindoas as linhas vazias da coluna 'Diagnostico Funcional' por "não informado" adaptando o código para lidar com linhas com espaços em branco e valores nulos
df['Diagnóstico Funcional'] = df['Diagnóstico Funcional'].str.strip().fillna('não informado')
print(df['Diagnóstico Funcional'].value_counts())

#corrigindo os textos com erros de digitação
df['Diagnóstico Funcional'] = df['Diagnóstico Funcional'].replace({
    'Dupla hemi ': 'Dupla Hemi'
})

Diagnóstico Funcional
não informado                                             213
Hemiparesia                                                46
Paraparesia                                                 6
Paraplegia                                                  2
Paraplegia espástica                                        2
Dupla hemiparesia                                           2
Deficiência visual                                          2
                                                            1
Tetraparesia espástica C5 ASIA D                            1
Tetraparesia                                                1
Dupla hemiparesia / Tetraparesia E deficiente auditivo      1
Dupla hemiparesia e deficiente visual                       1
Paraparesia flácida                                         1
Dupla hemi                                                  1
Dupla Hemi                                                  1
Tetraparesia incompleta                         

In [62]:
print(df['Nº quedas ao chão'].dtype)
print(df['Nº quedas ao chão'].value_counts())

#mudando a coluna 'Nº quedas ao chão' para o tipo inteiro, para que possamos fazer análises estatísticas
df['Nº quedas ao chão'] = pd.to_numeric(df['Nº quedas ao chão'], errors='coerce').astype('Int64')


Int64
Nº quedas ao chão
0     128
1      68
2      32
3      19
4       9
5       9
10      4
6       3
20      1
16      1
Name: count, dtype: Int64
